# Receipt Data Extraction and SQL Generation using Vision API

### Install Dependencies

In [ ]:
!pip -q install google-cloud-vision pillow pandas google-genai opencv-python addict --upgrade

### Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from google.colab import files
from google import genai

###Upload Images + Ground Truth File

In [ ]:
uploaded = files.upload()

image_files = []
ground_truth_file = None

for fname in uploaded.keys():
    if fname.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        image_files.append(fname)
    elif fname.lower().endswith(".json"):
        ground_truth_file = fname

image_files = sorted(image_files)

print("Images found:", image_files)
print("Ground truth file:", ground_truth_file)

if not image_files:
    raise ValueError("No image files uploaded.")

if ground_truth_file is None:
    raise ValueError("Please upload a ground_truth.json file.")


Saving receipt_01.jpg to receipt_01.jpg
Saving receipt_02.jpg to receipt_02.jpg
Saving receipt_03.jpg to receipt_03.jpg
Saving receipt_04.jpg to receipt_04.jpg
Saving receipt_05.jpg to receipt_05.jpg
Saving receipt_06.jpg to receipt_06.jpg
Saving receipt_07.jpg to receipt_07.jpg
Saving truth.json to truth.json
Images found: ['receipt_01.jpg', 'receipt_02.jpg', 'receipt_03.jpg', 'receipt_04.jpg', 'receipt_05.jpg', 'receipt_06.jpg', 'receipt_07.jpg']
Ground truth file: truth.json


###LOAD GROUND TRUTH

In [ ]:
with open(ground_truth_file, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

### Set Up Gemini Client

In [ ]:
GEMINI_API_KEY = "gemini-api-key"
client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"

In [ ]:
# Authenticate with Google Cloud Vision API
from google.colab import files
from google.cloud import vision
import os

# Upload service account JSON key
uploaded_auth = files.upload()
service_account_file = list(uploaded_auth.keys())[0]

# Set authentication
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = service_account_file

# Initialize Vision API client
vision_client = vision.ImageAnnotatorClient()
print(f"Vision API authenticated with {service_account_file}")

Saving ace-agility-235613-b3490f7b764d.json to ace-agility-235613-b3490f7b764d.json
✓ Vision API authenticated with ace-agility-235613-b3490f7b764d.json


### Image Preprocessing

In [ ]:
def preprocess_image(image_path: str) -> str:
    """
    Advanced preprocessing for handwritten / scanned text:
    - Convert to grayscale
    - Denoise with Gaussian blur
    - Adaptive threshold for uneven lighting and handwriting
    - Morphological cleanup
    - Save processed image
    Returns path to processed image
    """
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Denoise
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Adaptive threshold works better for uneven lighting and handwriting
    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    # Optional small morphological cleanup
    kernel = np.ones((2, 2), np.uint8)
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    processed_path = image_path.replace(".", "_processed.")
    cv2.imwrite(processed_path, cleaned)

    return processed_path

### OCR

In [ ]:
def run_ocr(image_path):
    """
    Extract text from image using Google Cloud Vision API TEXT_DETECTION.
    Preprocesses image first for better OCR accuracy.
    Returns raw OCR text.
    """
    from google.cloud import vision

    # Preprocess image (grayscale, denoise, adaptive threshold)
    processed_path = preprocess_image(image_path)

    # Read processed image file as bytes
    with open(processed_path, 'rb') as f:
        image_content = f.read()

    # Create Vision API request
    image = vision.Image(content=image_content)

    # Use TEXT_DETECTION for general handwritten/printed text
    response = vision_client.text_detection(image=image)

    # Extract full text from response
    if response.text_annotations:
        # First element is the full text, rest are individual words
        full_text = response.text_annotations[0].description
        return full_text.strip()
    else:
        return "[No text detected]"

### Call Gemini to parse to JSON

In [ ]:
import time

def extract_structured_data(ocr_text: str) -> str:
    prompt = f"""
You are an AI system that extracts structured data from handwritten Telugu receipts.
You are correcting noisy OCR from handwritten Telugu receipts.
COnvert telugu words to english.

Task:
1. Read the OCR text.
2. Correct obvious OCR mistakes.
3. Translate Telugu text into English.
4. Extract receipt rows into JSON.

The OCR text may contain:
- spelling mistakes
- broken words
- missing characters
- mixed Telugu and English

Return ONLY valid JSON.
No markdown.
No code fences.
The item name should be a single most probable word.
Price should be followed by .00

Use this schema:
[
  {{
    "item": "string",
    "qty": "string",
    "price": "string"
  }}
]

OCR TEXT:
{ocr_text}
"""

    max_retries = 5
    retry_delay = 2  # seconds

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config={"temperature": 0}
            )
            return response.text.strip()

        except Exception as e:
            error_str = str(e)

            # Check if it's a 503 error
            if "503" in error_str or "UNAVAILABLE" in error_str:
                if attempt < max_retries - 1:
                    wait_time = retry_delay * (2 ** attempt)  # exponential backoff
                    print(f"503 Error (attempt {attempt + 1}/{max_retries}). Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"Failed after {max_retries} attempts. Raising error.")
                    raise
            else:
                # For non-503 errors, raise immediately
                raise


### Convert JSON to SQL

In [ ]:
def json_to_sql(json_text: str, table_name: str = "receipts") -> str:
    # Clean markdown fences if Gemini returns them
    cleaned = json_text.replace("```json", "").replace("```", "").strip()

    data = json.loads(cleaned)

    sql_lines = []
    for row in data:
        item = str(row.get("item", "")).replace("'", "''")
        qty = str(row.get("qty", "")).replace("'", "''")
        price = str(row.get("price", "")).replace("'", "''")

        sql = f"INSERT INTO {table_name} (item_name, quantity, price) VALUES ('{item}', '{qty}', '{price}');"
        sql_lines.append(sql)

    return "\n".join(sql_lines)

### Accuracy check



In [ ]:
from difflib import SequenceMatcher

def evaluate_accuracy(image_name: str, generated_sql: str, ground_truth: dict) -> dict:
    """
    Evaluate accuracy using fuzzy string matching instead of substring matching.
    Handles spelling variations and similar item names.
    """
    expected_items = ground_truth.get(image_name, [])

    if not expected_items:
        return {
            "image": image_name,
            "expected_count": 0,
            "matched_count": 0,
            "accuracy": None
        }

    total = len(expected_items)
    matched = 0

    generated_lower = generated_sql.lower()

    for item in expected_items:
        expected_item = str(item.get("item", "")).lower().strip()

        if not expected_item:
            continue

        # Fuzzy matching: check if item appears as substring OR has high similarity
        if expected_item in generated_lower:
            # Exact substring match
            matched += 1
        else:
            # Fuzzy match: find best match in SQL output and check similarity
            words_in_sql = generated_lower.split()
            best_similarity = 0

            for word in words_in_sql:
                # Compare expected item with each word in SQL
                similarity = SequenceMatcher(None, expected_item, word).ratio()
                best_similarity = max(best_similarity, similarity)

            # If similarity >= 70%, count as match (handles spelling variations)
            if best_similarity >= 0.7:
                matched += 1

    accuracy = matched / total if total else None

    return {
        "image": image_name,
        "expected_count": total,
        "matched_count": matched,
        "accuracy": accuracy
    }

### Proccess Check

In [ ]:
results = []
sql_outputs = {}

for img_path in image_files:
    print("=" * 70)
    print(f"Processing: {img_path}")

    try:
        ocr_text = run_ocr(img_path)
        print("\n--- OCR TEXT ---")
        print(ocr_text if ocr_text else "[No text detected]")

        structured_json = extract_structured_data(ocr_text)
        print("\n--- GEMINI JSON ---")
        print(structured_json)

        sql_output = json_to_sql(structured_json)
        print("\n--- SQL OUTPUT ---")
        print(sql_output)

        sql_outputs[img_path] = sql_output

        acc = evaluate_accuracy(img_path, sql_output, ground_truth)

        results.append({
            "image": img_path,
            "ocr_text": ocr_text,
            "structured_json": structured_json,
            "sql_output": sql_output,
            "expected_count": acc["expected_count"],
            "matched_count": acc["matched_count"],
            "accuracy": acc["accuracy"]
        })

    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        results.append({
            "image": img_path,
            "ocr_text": "",
            "structured_json": "",
            "sql_output": "",
            "expected_count": 0,
            "matched_count": 0,
            "accuracy": None,
            "error": str(e)
        })

Processing: receipt_01.jpg

--- OCR TEXT ---
17-06-2016
-
450.00
200.00
-
300.00
కందిపప్పుల 3ky
సెసర పప్పు -2100
మినపప్పు 2
శనగపప్పు 1kg
ఆవాలు 1/2
జీలకర్ర 4lg
508534301248
ఇంగన డబ్బా
సబ్బులు -2 (త్రికూడు)
నూనె 3 leg.
-
100-00
40-00
55-00
30.00
45.00
50.00
48.00
270-00
80.00
నెయ్యి likeg .
-
125.00
పంచ ధర
leg
1793-00

--- GEMINI JSON ---
[
  {
    "item": "Kandipappu",
    "qty": "3kg",
    "price": "450.00"
  },
  {
    "item": "PesaraPappu",
    "qty": "2kg",
    "price": "200.00"
  },
  {
    "item": "Minapappu",
    "qty": "2kg",
    "price": "300.00"
  },
  {
    "item": "Senagapappu",
    "qty": "1kg",
    "price": "100.00"
  },
  {
    "item": "Avalu",
    "qty": "0.5kg",
    "price": "40.00"
  },
  {
    "item": "Jeelakarra",
    "qty": "1kg",
    "price": "55.00"
  },
  {
    "item": "Inguva",
    "qty": "1",
    "price": "30.00"
  },
  {
    "item": "Soaps",
    "qty": "2",
    "price": "45.00"
  },
  {
    "item": "Oil",
    "qty": "3kg",
    "price": "50.00"
  },
  {
    "it

### Summary Table

In [ ]:
df = pd.DataFrame(results)

# Convert accuracy to percentage for display
df["accuracy_percent"] = df["accuracy"].apply(
    lambda x: None if x is None else round(x * 100, 2)
)

display(df[["image", "expected_count", "matched_count", "accuracy_percent"]])

,image,expected_count,matched_count,accuracy_percent
0,receipt_01.jpg,13,4,30.77
1,receipt_02.jpg,5,3,60.00
2,receipt_03.jpg,13,10,76.92
3,receipt_04.jpg,4,4,100.00
4,receipt_05.jpg,4,4,100.00
5,receipt_06.jpg,4,2,50.00
6,receipt_07.jpg,5,3,60.00


### Overall Accuracy

In [ ]:
valid_accuracies = []
for r in results:
    acc = r.get("accuracy")
    if acc is not None and acc == acc:
        valid_accuracies.append(acc)

if valid_accuracies:
    overall_accuracy = sum(valid_accuracies) / len(valid_accuracies)
    print(f"\nOVERALL AVERAGE ACCURACY: {overall_accuracy * 100:.2f}%")
else:
    print("\nNo valid accuracy values found.")



OVERALL AVERAGE ACCURACY: 68.24%


Save Outputs

In [ ]:
with open("all_sql_outputs.txt", "w", encoding="utf-8") as f:
    for img, sql in sql_outputs.items():
        f.write(f"### {img}\n")
        f.write(sql + "\n\n")

df.to_csv("evaluation_results.csv", index=False)
